# Vector Database Foundations with ChromaDB — Step-by-Step Guide

**What is a Vector Database?**  
Traditional databases look for exact keyword matches. Vector databases search by **semantic meaning**.  
Text is transformed into high-dimensional numerical vectors (embeddings). Sentences sharing similar concepts, contexts, or meanings end up close together in vector space, allowing computers to search by idea rather than exact words.

---

### Vector Indexing & Semantic Search Architecture

```
┌────────────────────────────────────────────────────────────────────────┐
│                          INDEXING PHASE                                │
│                                                                        │
│  News Articles (Strings)         Embedding Model            ChromaDB   │
│  ┌───────────────────────┐      ┌───────────────┐          ┌─────────┐ │
│  │ "Tesla booked profit" │ ───► │ Dense Vector  │ ───────► │ Vector  │ │
│  │ "Apple leads market"  │      │ Representation│          │ Storage │ │
│  │ "SpaceX NASA contract"│      │ (e.g. 384 dim)│          │ Index   │ │
│  └───────────────────────┘      └───────────────┘          └─────────┘ │
└────────────────────────────────────────────────────────────────────────┘
                                      │
                                      ▼
┌────────────────────────────────────────────────────────────────────────┐
│                          RETRIEVAL PHASE                               │
│                                                                        │
│  User Semantic Query             Embedding Model          Distance     │
│  ┌───────────────────────┐      ┌───────────────┐         Calculation  │
│  │ "Query related to     │ ───► │ Query Vector  │ ──────► Cosine / L2  │
│  │  Elon Musk"           │      │               │         Comparison   │
│  └───────────────────────┘      └───────────────┘              │       │
│                                                                ▼       │
│                                                          Top-k Matches │
│                                                          [Tesla,SpaceX]│
└────────────────────────────────────────────────────────────────────────┘
```

**Key Concepts Covered:**
1. **ChromaDB Client Setup**: Working with in-memory ephemeral vector stores.
2. **Collection Lifecycle**: Creating collections safely (`get_or_create_collection`).
3. **Indexing Documents**: Supplying unique document IDs and textual payloads.
4. **Semantic Proximity Search**: Querying by conceptual intent without exact keyword overlap.


---
## Step 1 — Initialize the ChromaDB Client

`chromadb.Client()` initializes an in-memory client. For rapid experimentation and local testing, an in-memory client runs entirely in RAM without requiring an external database server.


In [1]:
import chromadb

client = chromadb.Client()


---
## Step 2 — Create Collection and Add Documents

**What is a Collection?**  
A collection in Chroma is analogous to a table in a relational database. It stores:
- **`documents`**: The raw text snippets.
- **`ids`**: Unique string identifiers for each record.
- **`embeddings`**: The generated vector arrays (Chroma generates these automatically if an embedding function is not explicitly passed).

> **Tip:** We use `get_or_create_collection` so running this cell multiple times won't raise a `Collection already exists` error.


In [2]:
collections = client.get_or_create_collection("news_articles")

collections.add(
    ids=["id1", "id2", "id3", "id4"],
    documents=[
        "Apple is leading in a smart phone game with iPhone sales up by 35%",
        "Tesla booked a minor profit of 1 billion $ in Q2",
        "Apples are high in fiber, vitamin C, and various antioxidants",
        "SpaceX got NASA contract worth 10 billion $",
    ]
)


---
## Step 3 — Query 1: Semantic Intent Matching ("Elon Musk")

Notice our query is:  
`"This is a query related to Elon Musk"`

*The text documents do NOT contain the words "Elon Musk".*  
Yet, the vector search correctly retrieves **Tesla** and **SpaceX** as the top results because their embeddings occupy a nearby region in semantic vector space.


In [3]:
results = collections.query(
    query_texts=["This is a query related to Elon Musk"],
    n_results=2
)
print(results)


---
## Step 4 — Query 2: Product & Context Disambiguation ("I love iPad")

Notice our query is:  
`"I love iPad"`

The vector search retrieves the articles relating to **Apple** (iPhone smartphone market and nutritional apples) based on contextual similarity.


In [4]:
results = collections.query(
    query_texts=["I love iPad"],
    n_results=2
)
print(results)
